# Track 1: YOLO for PCB Defect Detection (Sample / Pipeline Test Mode)
## Dual-Track Architecture -- Lightweight Edge CNN Branch

**Project:** FOD Predictive Detection / PCB Defect Detection  
**Model:** YOLO26n / YOLOv8n (~2.57M parameters)  
**Sample Test Mode:** Configured with `SAMPLE_MODE = True` to use ~50 training images and ~20 validation images for rapid pipeline verification (~30 seconds end-to-end).


## 1. Environment Verification & Hardware Detection


In [ ]:
!pip install -q ultralytics
import os
import sys
import torch
import ultralytics

print("=" * 60)
print("  Environment Verification")
print("=" * 60)
print(f"Python Version:   {sys.version.split()[0]}")
print(f"PyTorch Version:  {torch.__version__}")
print(f"Ultralytics:      {ultralytics.__version__}")

CUDA_AVAILABLE = torch.cuda.is_available()
if CUDA_AVAILABLE:
    DEVICE = 0
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print("CUDA Available:   YES [OK]")
    print(f"Active GPU:       {gpu_name} ({gpu_mem:.1f} GB VRAM)")
    RECOMMENDED_BATCH = 16
else:
    DEVICE = "cpu"
    print("CUDA Available:   NO (Running on CPU)")
    RECOMMENDED_BATCH = 8

print(f"Selected Device:  {DEVICE}")
print(f"Default Batch:    {RECOMMENDED_BATCH}")
print("=" * 60)


## 2. Dataset Configuration & Small Sample Extraction
- Mounts Google Drive (if running on Colab) or detects local folders.
- **`SAMPLE_MODE = True`**: Extracts ~50 training images and ~20 validation images across all 9 defect classes into `/content/dataset_sample`.
- When ready for full training, simply change `SAMPLE_MODE = False`!


In [ ]:
import os
import sys
import shutil
import yaml
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# ------------------------------------------------------------------------------
# ⚙️ CONFIGURATION & SAMPLE MODE TOGGLE
# ------------------------------------------------------------------------------
SAMPLE_MODE = True        # <--- Set to False when you want the FULL dataset!
MAX_TRAIN_SAMPLES = 50    # Number of training images in sample test mode
MAX_VAL_SAMPLES = 20      # Number of validation images in sample test mode

DRIVE_FOLDER_NAME = "Classified_Dataset"   # Name of your folder in Google Drive
# ------------------------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    MY_DRIVE = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else Path('/content/drive/My Drive')
    SOURCE_PATH = MY_DRIVE / DRIVE_FOLDER_NAME

    if not SOURCE_PATH.exists():
        # Check inside common subfolder
        sub = MY_DRIVE / "DataSet_CourseProject" / DRIVE_FOLDER_NAME
        if sub.exists():
            SOURCE_PATH = sub
        else:
            print(f"❌ Folder '{DRIVE_FOLDER_NAME}' not found at: {SOURCE_PATH}")
            print("\nFolders visible in your My Drive:")
            for folder in sorted([f.name for f in MY_DRIVE.iterdir() if f.is_dir()]):
                print(f"  📁 {folder}")
            raise FileNotFoundError("Please check your folder name in Google Drive.")

    BASE_DIR = Path("/content")
    LOCAL_DATASET = Path("/content/dataset_sample" if SAMPLE_MODE else "/content/dataset")
else:
    BASE_DIR = Path.cwd()
    SOURCE_PATH = Path.cwd() / "Data Preprocessing" / "Classified_Dataset"
    if not SOURCE_PATH.exists():
        SOURCE_PATH = Path.cwd() / "Dataset" / "DsPCBSD+" / "Data_YOLO"
    LOCAL_DATASET = Path.cwd() / ("dataset_sample" if SAMPLE_MODE else "dataset_full")

print(f"✅ Source Dataset located: {SOURCE_PATH}")
print(f"📁 Target Local Dataset:  {LOCAL_DATASET} (SAMPLE_MODE={SAMPLE_MODE})")

# Clean target folder if in sample mode to ensure fresh exact sample count
if SAMPLE_MODE and LOCAL_DATASET.exists():
    shutil.rmtree(LOCAL_DATASET)

LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# Extract / Cache Images and Matched Labels
# ------------------------------------------------------------------------------
for split in ['train', 'val']:
    target_img_dir = LOCAL_DATASET / 'images' / split
    target_lbl_dir = LOCAL_DATASET / 'labels' / split
    target_img_dir.mkdir(parents=True, exist_ok=True)
    target_lbl_dir.mkdir(parents=True, exist_ok=True)

    src_img = SOURCE_PATH / 'images' / split
    src_lbl = SOURCE_PATH / 'labels' / split

    limit = MAX_TRAIN_SAMPLES if split == 'train' else MAX_VAL_SAMPLES

    if SAMPLE_MODE:
        # Check if source has class subdirectories (e.g. Classified_Dataset)
        class_dirs = [d for d in src_img.iterdir() if d.is_dir()]
        copied_count = 0

        if class_dirs:
            # Distribute sample evenly across all classes so all 9 classes are represented
            per_class = max(1, limit // len(class_dirs))
            for c_dir in class_dirs:
                c_imgs = list(c_dir.glob("*.jpg"))[:per_class]
                for img_file in c_imgs:
                    dst_img = target_img_dir / img_file.name
                    if not dst_img.exists():
                        shutil.copy2(str(img_file), str(dst_img))
                        copied_count += 1
                    # Find corresponding label in src_lbl
                    matched_lbls = list(src_lbl.rglob(img_file.stem + ".txt"))
                    if matched_lbls:
                        shutil.copy2(str(matched_lbls[0]), str(target_lbl_dir / matched_lbls[0].name))

            # If under limit, top up from any remaining images
            if copied_count < limit:
                for img_file in src_img.rglob("*.jpg"):
                    if copied_count >= limit:
                        break
                    dst_img = target_img_dir / img_file.name
                    if not dst_img.exists():
                        shutil.copy2(str(img_file), str(dst_img))
                        copied_count += 1
                        matched_lbls = list(src_lbl.rglob(img_file.stem + ".txt"))
                        if matched_lbls:
                            shutil.copy2(str(matched_lbls[0]), str(target_lbl_dir / matched_lbls[0].name))
        else:
            # Flat source structure
            for img_file in list(src_img.glob("*.jpg"))[:limit]:
                shutil.copy2(str(img_file), str(target_img_dir / img_file.name))
                matched_lbls = list(src_lbl.glob(img_file.stem + ".txt"))
                if matched_lbls:
                    shutil.copy2(str(matched_lbls[0]), str(target_lbl_dir / matched_lbls[0].name))
    else:
        # FULL DATASET MODE: Copy and flatten all images & labels
        if len(list(target_img_dir.glob("*.jpg"))) < 1000:
            print(f"  Copying full {split} split to fast local disk...")
            for img_file in src_img.rglob("*.jpg"):
                shutil.copy2(str(img_file), str(target_img_dir / img_file.name))
            for lbl_file in src_lbl.rglob("*.txt"):
                shutil.copy2(str(lbl_file), str(target_lbl_dir / lbl_file.name))

    n_imgs = len(list(target_img_dir.glob("*.jpg")))
    n_lbls = len(list(target_lbl_dir.glob("*.txt")))
    print(f"  [{split:5s}]: {n_imgs} images, {n_lbls} matched label files prepared.")

# ------------------------------------------------------------------------------
# Automatically generate pcb_data.yaml
# ------------------------------------------------------------------------------
CLASS_NAMES = {
    0: "SH",    # Solder Pads
    1: "SP",    # Shorts
    2: "SC",    # Scratches
    3: "OP",    # Opening
    4: "MB",    # Mask Break
    5: "HB",    # Hole Break
    6: "CS",    # Cremation
    7: "CFO",   # Component Foreign Object
    8: "BMFO",  # Bare Metal-Free Offset
}

data_cfg = {
    "path": LOCAL_DATASET.as_posix(),
    "train": "images/train",
    "val": "images/val",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES
}

YAML_PATH = LOCAL_DATASET / "pcb_data.yaml"
with open(YAML_PATH, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

# Export standard global paths for subsequent cells
dataset_root = LOCAL_DATASET
train_img_dir = dataset_root / "images" / "train"
val_img_dir = dataset_root / "images" / "val"

print("\n" + "=" * 60)
print(f"✅ Configuration File: {YAML_PATH}")
print(f"Train path: {train_img_dir} ({len(list(train_img_dir.glob('*.jpg')))} files)")
print(f"Val path:   {val_img_dir} ({len(list(val_img_dir.glob('*.jpg')))} files)")
print("=" * 60)


## 3. Dataset Visualization -- Ground Truth Samples


In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

sample_images = list(train_img_dir.glob("*.jpg"))[:4]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, img_path in enumerate(sample_images):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    ax = axes[idx]
    ax.imshow(img_rgb)
    ax.set_title(img_path.name, fontsize=10)
    ax.axis("off")

    lbl_path = dataset_root / "labels" / "train" / (img_path.stem + ".txt")
    if lbl_path.exists():
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    x1 = (xc - bw / 2) * w
                    y1 = (yc - bh / 2) * h
                    box_w = bw * w
                    box_h = bh * h

                    rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)
                    ax.text(x1, max(y1 - 4, 0), CLASS_NAMES.get(cls_id, str(cls_id)),
                            color='yellow', fontsize=8, fontweight='bold',
                            bbox=dict(facecolor='black', alpha=0.6, pad=1))

plt.suptitle(f"Sample Ground-Truth PCB Defect Annotations ({'Sample Mode' if SAMPLE_MODE else 'Full Dataset'})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Track 1 Model Initialization (YOLO26n / YOLOv8n)


In [ ]:
from ultralytics import YOLO

MODEL_WEIGHTS = "yolo26n.pt"
print(f"Loading pretrained baseline: {MODEL_WEIGHTS} ...")
model = YOLO(MODEL_WEIGHTS)

print(f"Loaded architecture: {model.model.__class__.__name__}")
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total Parameters: {total_params:,} (~{total_params/1e6:.2f}M)")


## 5. Training Execution
- In **Sample Mode**: Trains for **5 epochs** on the ~50 images (~20–30 seconds on GPU, ~1 minute on CPU) to verify the full forward/backward training loop.
- In **Full Mode**: Trains for **100 epochs** with early stopping (`patience=20`).


In [ ]:
# Configure training parameters
EPOCHS = 5 if SAMPLE_MODE else 100       # 5 epochs for smoke test, 100 for full convergence
BATCH_SIZE = 8 if SAMPLE_MODE else RECOMMENDED_BATCH
IMG_SIZE = 640
PROJECT_DIR = str(BASE_DIR / "runs" / "detect")
RUN_NAME = "PCB_YOLO_SampleTest" if SAMPLE_MODE else "PCB_YOLO26n_Run"

print("=" * 60)
print(f"  Starting YOLO Training ({'SAMPLE SMOKE TEST' if SAMPLE_MODE else 'FULL TRAINING'})")
print("=" * 60)
print(f"  Data Config:    {YAML_PATH}")
print(f"  Device:         {DEVICE}")
print(f"  Batch Size:     {BATCH_SIZE}")
print(f"  Image Size:     {IMG_SIZE}")
print(f"  Epochs:         {EPOCHS}")
print(f"  Output Dir:     {PROJECT_DIR}/{RUN_NAME}")
print("=" * 60)

results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2,
    project=PROJECT_DIR,
    name=RUN_NAME,
    patience=20,
    save=True,
    plots=True,
    amp=(DEVICE != "cpu"),
    exist_ok=True
)

BEST_WEIGHTS = Path(results.save_dir) / "weights" / "best.pt"
if not BEST_WEIGHTS.exists():
    BEST_WEIGHTS = Path(results.save_dir) / "weights" / "last.pt"

print(f"\n✅ Training completed! Checkpoint saved to: {BEST_WEIGHTS}")


## 6. Model Validation & Per-Class Metrics Evaluation


In [ ]:
import os
import pandas as pd
from ultralytics import YOLO

# 1. Load your real trained checkpoint
real_checkpoint = "/content/runs/detect/PCB_YOLO26n_Run/weights/best.pt"
if not os.path.exists(real_checkpoint):
    real_checkpoint = BEST_WEIGHTS if "BEST_WEIGHTS" in locals() and BEST_WEIGHTS.exists() else "yolo26n.pt"

eval_model = YOLO(str(real_checkpoint))
print(f"Evaluating model checkpoint: {real_checkpoint}")

val_results = eval_model.val(
    data=str(YAML_PATH),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE
)

print("\n" + "=" * 60)
print("  Validation Performance Summary")
print("=" * 60)
print(f"  mAP@50:       {val_results.box.map50:.4f}")
print(f"  mAP@50-95:    {val_results.box.map:.4f}")
print(f"  Precision:     {val_results.box.mp:.4f}")
print(f"  Recall:        {val_results.box.mr:.4f}")
print("=" * 60)

names = eval_model.names
class_metrics = []
for i in range(len(names)):
    ap50 = val_results.box.ap50[i] if len(val_results.box.ap50) > i else 0.0
    ap = val_results.box.ap[i] if len(val_results.box.ap) > i else 0.0
    class_metrics.append({
        "Class ID": i,
        "Class Name": names[i],
        "mAP@50": round(float(ap50), 4),
        "mAP@50-95": round(float(ap), 4)
    })

df_metrics = pd.DataFrame(class_metrics)
print("\nPer-Class Detection Accuracy:")
print(df_metrics.to_string(index=False))


## 7. Qualitative Predictions on Validation Images


In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
from PIL import Image

sample_val_imgs = list((LOCAL_DATASET / "images" / "val").glob("*.jpg"))
chosen_samples = random.sample(sample_val_imgs, min(4, len(sample_val_imgs)))

fig, axes = plt.subplots(1, min(4, len(chosen_samples)), figsize=(20, 5))
if len(chosen_samples) == 1:
    axes = [axes]

for idx, img_p in enumerate(chosen_samples):
    pred = eval_model.predict(source=str(img_p), conf=0.25, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    plotted = pred.plot()
    plotted_rgb = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)

    ax = axes[idx]
    ax.imshow(plotted_rgb)
    ax.set_title(f"{img_p.name}\n({len(pred.boxes)} detections)", fontsize=10)
    ax.axis("off")

plt.suptitle("YOLO Track 1 Real Predictions (Confidence >= 0.25)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 8. Model Export for Edge Deployment (ONNX / FP16)


In [ ]:
import os
import shutil

print("Exporting trained YOLO model to ONNX format...")
try:
    onnx_path = eval_model.export(format="onnx", imgsz=IMG_SIZE, simplify=True)
    print(f"✅ Export successful! ONNX model saved at: {onnx_path}")

    size_pt = os.path.getsize(str(real_checkpoint)) / (1024 * 1024)
    size_onnx = os.path.getsize(str(onnx_path)) / (1024 * 1024)
    print(f"PyTorch size:  {size_pt:.2f} MB")
    print(f"ONNX size:     {size_onnx:.2f} MB ({(1 - size_onnx/size_pt)*100:.1f}% reduction)")
except Exception as e:
    print(f"Export note: {e}")

print("\nSyncing final ONNX model and plots to Google Drive...")
try:
    shutil.copytree("/content/runs", "/content/drive/MyDrive/DataSet_CourseProject/runs_backup", dirs_exist_ok=True)
    print("✅ ALL TRACK 1 ARTIFACTS PERMANENTLY SAVED TO GOOGLE DRIVE!")
except Exception as e:
    print(f"Drive sync note: {e}")
